# 🏷️ CRM Entity Recognition Pipeline
### Hybrid NER System — GLiNER + Regex + Rules + Post-Processing

**What this notebook does:**  
Automatically extracts structured information (entities) from raw sales emails and CRM chat messages.  
Instead of reading every email manually, the pipeline identifies *who* is involved, *what company* they work at,  
*what budget* they have, *when* they want to meet, and much more — automatically.

---

**Entities extracted:**

| Label | Example |
|---|---|
| `person` | *James Whitfield*, *Sarah* |
| `company` | *NovaTech Solutions*, *Meridian Group* |
| `job title` | *Head of Operations*, *VP of Sales* |
| `email address` | *james@novatech.ae* |
| `phone number` | *+971 50 123 4567* |
| `date` | *Thursday, June 12th*, *Q3*, *next week* |
| `money` | *$45,000*, *120k*, *$4,500/month* |
| `location` | *Dubai*, *Southeast Asia* |
| `product` | *Enterprise Suite*, *CRM platform* |
| `competitor` | *Salesforce*, *HubSpot* |

---

**Architecture at a glance:**
```
Raw Text
   │
   ├─► GLiNER Large (urchade/gliner_large-v2.1)  ─┐
   ├─► GLiNER Medium (urchade/gliner_medium-v2.1) ─┤─ Ensemble union
   │                                               ─┘
   ├─► Regex: email, phone, money, date
   ├─► Competitor context rules
   │
   └─► Post-Processing:
         1. Per-label confidence thresholds
         2. Merge adjacent spans
         3. Cross-entity validation (email domain → not a company)
         4. Product context filter
         5. Pronoun filters (We/They → not company; I/Hi → not person)
         6. Money word filter (Budget/invoice → not money)
         7. Location filter (phone digits → not location)
         8. Job-title non-phrase filter
         9. Product single-word filter
        10. Containment dedup (June 12th dropped when Thursday, June 12th exists)
```

**Final benchmark results (150 test cases, 10 labels):**  
Recall **95.6%** · Precision **91.9%** · F1 **0.937** · 


In [ ]:
# Install required packages (run once)
# gliner  — zero-shot NER model
# rich    — pretty console output (optional in Colab, used for local runs)
!pip install -q gliner rich
print("✓ Packages installed")

## 1 · Imports & Setup

In [ ]:
import re, json, time, base64, zlib
from dataclasses import dataclass
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# Colab display helper
def show(df, title=""):
    if title:
        display(HTML(f"<h4>{title}</h4>"))
    display(df)

print("✓ Imports ready")

## 2 · Entity Configuration

### 2.1 Labels & Confidence Thresholds

GLiNER outputs a confidence score (0–1) for each candidate entity.  
We set a **different threshold per label** based on how often each label produces false positives:

- **Low threshold** (0.35) for labels where the model is reliable and missing one is costly  
- **Higher threshold** (0.55) for labels that tend to hallucinate common words

The regex-based extractors (email, phone, money, date) bypass thresholds entirely —  
they either match a pattern or they don't.

In [ ]:
GLINER_LABELS = [
    "person", "company", "job title", "date",
    "money", "location", "product", "competitor"
]

LABEL_THRESHOLDS = {
    "person":     0.55,   # cuts single-word low-confidence names
    "company":    0.55,   # cuts email-fragment companies
    "job title":  0.50,   # cuts ambiguous single-word extractions
    "date":       0.35,   # low — GLiNER dates are generally reliable
    "money":      0.45,   # cuts vague financial terms
    "location":   0.50,   # moderate — avoids generic words
    "product":    0.50,   # raised from 0.30 in v1 — product was noisy
    "competitor": 0.35,   # low — competitor recall was the hardest to achieve
}

# Display thresholds
df_thresh = pd.DataFrame([
    {"Label": k, "GLiNER threshold": v, "Source": "GLiNER model"}
    for k, v in LABEL_THRESHOLDS.items()
] + [
    {"Label": "email address", "GLiNER threshold": "N/A — regex", "Source": "Regex pattern"},
    {"Label": "phone number",  "GLiNER threshold": "N/A — regex", "Source": "Regex pattern"},
])
show(df_thresh.style.set_caption("Per-label confidence thresholds"))

### 2.2 Blocklists

Each blocklist targets a specific class of false positive that thresholds alone can't catch:

| Blocklist | Why it exists |
|---|---|
| **Company pronouns** | GLiNER sometimes labels *"We"*, *"They"*, *"us"* as companies |
| **Person pronouns** | GLiNER sometimes labels *"I"*, *"She"*, *"Hi"* as person names |
| **Location generics** | *"address"*, *"3 offices"* are not locations |
| **Product single words** | *"pilot"*, *"product"*, *"plan"* alone are not products |
| **Job-title non-titles** | *"Our team"*, *"Contact"*, *"follow-up"* are not job titles |

In [ ]:
# Words that GLiNER hallucinates as company names
_COMPANY_PRONOUN_BLOCKLIST = {
    "we", "i", "they", "it", "our", "your", "their", "us", "you",
    "he", "she", "this", "that", "these", "those", "his", "her",
    "internal", "company", "the client", "them",
}

# Pronouns falsely tagged as person names
_PERSON_PRONOUN_BLOCKLIST = {
    "they", "she", "he", "i", "them", "hi", "we", "his", "her", "you",
    "someone", "anyone", "whoever", "nobody", "somebody",
}

# Generic words falsely tagged as locations
_LOCATION_GENERIC_WORDS = {
    "address", "location", "locations", "office", "offices",
    "site", "sites", "internal", "remote",
}

# Single generic words falsely tagged as products
_PRODUCT_SINGLE_BLOCKLIST = {
    "product", "pilot", "platform", "tool", "suite", "solution", "solutions",
    "software", "system", "systems", "module", "bundle", "package",
    "service", "services", "reporting",
}

# Phrases falsely tagged as job titles
_JOB_TITLE_NONTITLE_PHRASES = {
    "contact", "our team", "main contact", "follow-up", "support team",
    "the team", "our staff", "my team",
}
_JOB_TITLE_NONTITLE_CONTAINS = [
    "poc is", "main contact", "follow-up", "our team",
]

print("✓ Blocklists defined")

## 3 · Regex Extractors

For 4 entity types, we skip GLiNER entirely and use hand-crafted regex patterns instead.

**Why regex for these labels?**
- **Email address** — GLiNER scored only 26% recall on emails in our baseline benchmark.  
  A simple email regex achieves 100% with zero false positives.
- **Phone number** — Same story. Structured patterns are far more reliable.
- **Money** — GLiNER doesn't know about `AED`, `EGP`, `SAR` currencies and misses `/month` suffixes.  
  Our regex handles currency symbols, codes, k/M/B suffixes, and `/period` formats.
- **Date** — Relative expressions like *"this Thursday"*, *"Q3"*, *"end of month"* are missed by GLiNER  
  but are easy to capture with patterns.

**Competitor** uses a hybrid: context patterns (did they say *"comparing you against..."*?)  
cross-referenced with a known-competitor name list.

In [ ]:
# ── Email ─────────────────────────────────────────────────────────────────────
_EMAIL_RE = re.compile(r'\b[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}\b')

# ── Phone ─────────────────────────────────────────────────────────────────────
# Matches international formats: +971 50 123 4567 / 01-555-1234 / +1-800-555-0199
_PHONE_RE = re.compile(r'(?<!\d)(\+?(?:\d[\s\-.]?){7,14}\d)(?!\d)')

# ── Money ─────────────────────────────────────────────────────────────────────
# Handles: $45,000  €1.2M  AED 500k  120,000 EGP  $4,500/month  six figures
_MONEY_PATTERNS = [
    re.compile(r'(?:[$€£¥])\s?\d[\d,]*(?:\.\d+)?(?:\s?(?:k|K|M|B|million|billion))?(?:/\w+)*', re.I),
    re.compile(r'\d[\d,]*(?:\.\d+)?\s?(?:k|K)\b'),
    re.compile(r'(?:USD|EUR|GBP|AED|SAR|EGP|GHS|CHF|JPY|CNY)\s?\d[\d,]*(?:\.\d+)?(?:\s?(?:k|K|M|B|million|billion))?', re.I),
    re.compile(r'\d[\d,]*(?:\.\d+)?\s?(?:USD|EUR|GBP|AED|SAR|EGP|GHS)', re.I),
    re.compile(r'(?:approximately\s+)?\d+(?:\.\d+)?\s+million\s+(?:EGP|USD|EUR|GBP|AED|SAR)', re.I),
    re.compile(r'(?:six|seven|eight|nine)\s+figures', re.I),
    re.compile(r'\d+\s+grand\b', re.I),
]

# ── Date ──────────────────────────────────────────────────────────────────────
# Handles: June 12th  Thursday, June 12th  Q3  this week  end of month  tomorrow
_MONTHS = r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
_DATE_PATTERNS = [
    re.compile(rf'{_MONTHS}\s+\d{{1,2}}(?:st|nd|rd|th)?(?:,?\s+\d{{4}})?', re.I),
    re.compile(rf'\d{{1,2}}(?:st|nd|rd|th)?\s+(?:of\s+)?{_MONTHS}(?:,?\s+\d{{4}})?', re.I),
    re.compile(r'\d{1,2}/\d{1,2}/\d{2,4}'),
    re.compile(r'Q[1-4]\s*(?:FY)?\s*\d{4}', re.I),
    re.compile(r'Q[1-4]\b', re.I),
    re.compile(r'(?:next|last|this|early next)\s+(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|week|month|quarter|year)', re.I),
    re.compile(r'(?:end of|by end of)\s+(?:the\s+)?(?:month|year|week|quarter|fiscal year|Friday|March|April|May|June|July|August|September|October|November|December|January|February|Q[1-4])', re.I),
    re.compile(r'(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)', re.I),
    re.compile(r'tomorrow|yesterday', re.I),
    re.compile(rf'(?:next|last)\s+{_MONTHS}', re.I),
]

# ── Competitors ───────────────────────────────────────────────────────────────
# Context signals: "comparing you against X", "switching from X", "vs X"
_COMP_PATTERNS = [
    re.compile(p, re.IGNORECASE) for p in [
        r'compar(?:ing|ed|e)?\s+(?:you\s+)?(?:against|with|to)\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:[,\n\.]|$| and )',
        r'evaluat(?:ing|ed|e)?\s+(?:your\s+)?(?:product\s+)?(?:against\s+)?(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:\s+(?:and|as|but|,)|$)',
        r'(?:switch(?:ed|ing)?|mov(?:ed?|ing)|migrat(?:ed?|ing))\s+(?:away\s+)?from\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:\s+to\b)',
        r'(?:went|go(?:ing)?|chose?)\s+with\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:[,.\n]|$)',
        r'(?:vs\.?|versus)\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:[,.\n]|$)',
        r'(?:we\'ve\s+(?:been\s+)?using|we\s+(?:currently\s+)?use|used)\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)\s+(?:for|and|but)',
        r'(?:alongside|alternative\s+to|instead\s+of|replace(?:ment)?(?:\s+for)?)\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:[,.\n]|$| and )',
        r'(?:away\s+from)\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:\s+and\s+(?P<name2>[A-Z][\w\s&.\-]{1,40}?))?',
        r'(?:they\s+(?:chose|went)\s+(?:with\s+)?)(?P<name>[A-Z][\w\s&.\-]{1,40}?)(?:[,.\n]|$)',
        r'(?:were?\s+with)\s+(?P<name>[A-Z][\w\s&.\-]{1,40}?)\s+(?:for|but)',
    ]
]

_KNOWN_COMPETITORS = {
    "salesforce","hubspot","zoho","zoho crm","pipedrive","freshsales",
    "microsoft dynamics","dynamics 365","monday crm","copper","oracle","sap",
    "salesforce sales cloud","microsoft dynamics 365 sales","sugar crm","fusion crm",
}

_PRODUCT_SIGNALS = re.compile(
    r'\b(?:plan|tier|package|subscription|module|suite|add[\-\s]?on|upgrade|downgrade|'
    r'version|edition|bundle|platform|tool|product)\b', re.I
)

print("✓ Regex patterns compiled")

### 3.1 — Regex Demo

Let's run each extractor on a sample text to see what they capture:

In [ ]:
DEMO_TEXT = (
    "Hi Sarah,\n\n"
    "I'm James Whitfield, Head of Operations at NovaTech Solutions.\n"
    "We're comparing you against Salesforce and HubSpot.\n"
    "Our budget is around $45,000 annually or $4,500/month.\n"
    "Could we schedule a call this Thursday, June 12th?\n"
    "Reach me at james.whitfield@novatech.ae or +971 50 123 4567.\n"
    "Best, James"
)

def demo_regex(text):
    results = {}

    # Email
    results["email address"] = [m.group() for m in _EMAIL_RE.finditer(text)]

    # Phone
    results["phone number"] = [m.group().strip() for m in _PHONE_RE.finditer(text)
                                if len(re.sub(r'\D','',m.group())) >= 7]

    # Money
    money, seen = [], set()
    for pat in _MONEY_PATTERNS:
        for m in pat.finditer(text):
            t = m.group().strip()
            if t not in seen:
                money.append(t); seen.add(t)
    results["money"] = money

    # Date
    dates, seen = [], set()
    for pat in _DATE_PATTERNS:
        for m in pat.finditer(text):
            t = m.group().strip()
            if t not in seen and len(t) > 2:
                dates.append(t); seen.add(t)
    results["date"] = dates

    return results

found = demo_regex(DEMO_TEXT)
print("Input text:")
print(DEMO_TEXT)
print()
print("Regex extractions:")
for label, items in found.items():
    print(f"  {label:<18} {items}")

## 4 · GLiNER Models

### What is GLiNER?

[GLiNER](https://github.com/urchade/GLiNER) (Generalist Lightweight model for Named Entity Recognition) is a zero-shot NER model.  
Unlike traditional NER models that have a fixed set of labels baked in during training,  
GLiNER accepts **any label you give it at inference time**.

We use it like this:
```python
entities = model.predict_entities(text, ["person", "company", "job title", ...], threshold=0.30)
```

### Why two models (ensemble)?

Running both the `large` and `medium` variants and **taking the union** of their predictions:
- The large model has better understanding of complex sentences
- The medium model sometimes catches entities the large model misses
- Together they achieve higher recall than either alone

*Trade-off: 2× inference time. Average latency ~880ms per document.*

In [ ]:
from gliner import GLiNER

print("Loading GLiNER large model (urchade/gliner_large-v2.1)...")
model_lg = GLiNER.from_pretrained("urchade/gliner_large-v2.1")
print("✓ Large model ready")

print("Loading GLiNER medium model (urchade/gliner_medium-v2.1)...")
model_md = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
print("✓ Medium model ready")

### 4.1 — GLiNER Demo

Let's see what the ensemble extracts from our sample text *before* any post-processing:

In [ ]:
ents_lg = model_lg.predict_entities(DEMO_TEXT, GLINER_LABELS, threshold=0.30)
ents_md = model_md.predict_entities(DEMO_TEXT, GLINER_LABELS, threshold=0.30)

print(f"Large model found {len(ents_lg)} entities:")
for e in ents_lg:
    print(f"  {e['label']:<18} '{e['text']}'  (score={e['score']:.2f})")

print(f"\nMedium model found {len(ents_md)} entities:")
for e in ents_md:
    print(f"  {e['label']:<18} '{e['text']}'  (score={e['score']:.2f})")

## 5 · Post-Processing Pipeline

After combining GLiNER + regex outputs, we apply **10 refinement steps** in order.  
Each step targets a specific class of noise discovered during benchmarking.

### Overview

| Step | What it does | Problem it solves |
|---|---|---|
| 1 | Threshold filter | Remove low-confidence GLiNER guesses |
| 2 | Exact dedup | Remove identical `(label, text)` pairs |
| 3 | Adjacent span merge | *"James"* + *"Whitfield"* → *"James Whitfield"* |
| 4 | Cross-entity validation | Email domain base can't be a company |
| 5 | Product context filter | *"platform"* kept only if near product signal words |
| 6 | Pronoun company filter | *"We"*, *"They"* → not a company |
| 7 | Person pronoun filter | *"I"*, *"She"*, *"Hi"* → not a person |
| 8 | Money word filter | *"Budget"*, *"invoice"* → no digits = not money |
| 9 | Location filter | Phone-digit strings & *"address"* → not a location |
| 10 | Job title filter | *"Contact"*, *"Our team"* → not a job title |
| 11 | Product word filter | Single generic words → not a product |
| 12 | **Containment dedup** | *"June 12th"* dropped when *"Thursday, June 12th"* exists |

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Step 1 — Per-label confidence threshold
# ────────────────────────────────────────────────────────────────────────────
def apply_label_thresholds(entities):
    return [e for e in entities
            if e.get("score", 1.0) >= LABEL_THRESHOLDS.get(e["label"], 0.4)]


# ────────────────────────────────────────────────────────────────────────────
# Step 2 — Exact deduplication: same (label, text) → keep first occurrence
# ────────────────────────────────────────────────────────────────────────────
def deduplicate(entities):
    seen, result = set(), []
    for e in entities:
        key = (e["label"], e["text"].lower().strip())
        if key not in seen:
            seen.add(key); result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 3 — Merge adjacent spans of the same label
# Example: "James" at pos 5 + "Whitfield" at pos 11 → "James Whitfield"
# ────────────────────────────────────────────────────────────────────────────
def merge_adjacent_spans(entities, text):
    if not entities: return entities
    positioned = []
    for e in entities:
        idx = text.lower().find(e["text"].lower())
        positioned.append((idx, e))
    positioned.sort(key=lambda x: (x[1]["label"], x[0]))
    merged = []
    i = 0
    while i < len(positioned):
        pos, ent = positioned[i]
        if pos < 0:
            merged.append(ent); i += 1; continue
        j = i + 1
        while j < len(positioned):
            pos2, ent2 = positioned[j]
            if ent2["label"] != ent["label"] or pos2 < 0: break
            gap = pos2 - (pos + len(ent["text"]))
            if 0 <= gap <= 4:
                combined = text[pos:pos2 + len(ent2["text"])]
                ent = {"label": ent["label"], "text": combined,
                       "score": min(ent.get("score",1.0), ent2.get("score",1.0))}
                j += 1
            else: break
        merged.append(ent); i = j
    return merged


# ────────────────────────────────────────────────────────────────────────────
# Step 4 — Cross-entity validation
# • Email local part (james.whitfield) → not a person name
# • Email domain base (novatech) → not a company
# • Company text that is a substring of a person name → likely wrong
# ────────────────────────────────────────────────────────────────────────────
def cross_entity_validate(entities):
    email_locals, email_domain_bases = set(), set()
    for e in entities:
        if e["label"] == "email address" and "@" in e["text"]:
            local, domain = e["text"].lower().split("@", 1)
            email_locals.add(local)
            email_domain_bases.add(domain.split(".")[0])
    person_texts = {e["text"].lower() for e in entities if e["label"] == "person"}
    result = []
    for e in entities:
        if e["label"] == "person" and e["text"].lower() in email_locals:
            continue
        if e["label"] == "company":
            tl = e["text"].lower().strip()
            if tl in email_locals or tl in email_domain_bases: continue
            if any(tl in p for p in person_texts): continue
        result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 5 — Product context filter
# A product entity is only kept if a product-signal word appears nearby in text
# (e.g. "plan", "suite", "module"). Score >= 0.6 bypasses this check.
# ────────────────────────────────────────────────────────────────────────────
def filter_products_by_context(entities, text):
    if not _PRODUCT_SIGNALS.search(text):
        return [e for e in entities if e["label"] != "product"]
    result = []
    for e in entities:
        if e["label"] != "product": result.append(e); continue
        idx = text.lower().find(e["text"].lower())
        if idx < 0: result.append(e); continue
        window = text[max(0, idx-60):idx+len(e["text"])+60].lower()
        if _PRODUCT_SIGNALS.search(window) or e.get("score", 1.0) >= 0.6:
            result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 6 — Pronoun company filter
# GLiNER hallucinates "We", "They", "us", "internal" as companies ~29% of the time
# ────────────────────────────────────────────────────────────────────────────
def filter_pronoun_companies(entities):
    return [e for e in entities
            if not (e["label"] == "company"
                    and e["text"].lower().strip() in _COMPANY_PRONOUN_BLOCKLIST)]


# ────────────────────────────────────────────────────────────────────────────
# Step 7 — Person pronoun filter
# "They", "She", "I", "Hi" are not person names
# ────────────────────────────────────────────────────────────────────────────
def filter_person_pronouns(entities):
    result = []
    for e in entities:
        if e["label"] == "person":
            words = e["text"].strip().split()
            if len(words) == 1 and e["text"].lower() in _PERSON_PRONOUN_BLOCKLIST:
                continue
        result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 8 — Money word filter
# Drop money entities with NO digits AND NO currency symbol ($€£¥)
# "Budget", "invoice", "fiscal year" → not money
# "$4,500", "500K", "EGP 80000" → kept
# ────────────────────────────────────────────────────────────────────────────
def filter_money_no_digits(entities):
    _symbols = set('$€£¥')
    result = []
    for e in entities:
        if e["label"] == "money":
            if (not any(c.isdigit() for c in e["text"])
                    and not any(c in _symbols for c in e["text"])):
                continue
        result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 9 — Location refined filter
# • >40% digit characters → phone number misclassified as location
# • Exact match to generic non-place word → not a location
# • "3 locations", "2 offices" → also not locations
# ────────────────────────────────────────────────────────────────────────────
def filter_location_refined(entities):
    result = []
    for e in entities:
        if e["label"] == "location":
            text = e["text"].strip()
            tl = text.lower()
            digits = sum(1 for c in text if c.isdigit())
            if len(text) > 0 and digits / len(text) > 0.40: continue
            if tl in _LOCATION_GENERIC_WORDS: continue
            words = tl.split()
            if (len(words) == 2
                    and words[0].isdigit()
                    and words[1].rstrip('s') in _LOCATION_GENERIC_WORDS):
                continue
        result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 10 — Job title non-phrase filter
# "Contact", "Our team", "follow-up" are not job titles
# NOTE: single words like "IT", "CEO", "ops" ARE valid job titles — not filtered
# ────────────────────────────────────────────────────────────────────────────
def filter_job_title_nontitles(entities):
    result = []
    for e in entities:
        if e["label"] == "job title":
            tl = e["text"].lower().strip()
            if tl in _JOB_TITLE_NONTITLE_PHRASES: continue
            if any(phrase in tl for phrase in _JOB_TITLE_NONTITLE_CONTAINS): continue
        result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 11 — Product single-word filter
# "pilot", "product", "plan" alone are not products
# Multi-word phrases ("CRM platform", "Enterprise Suite") are kept
# ────────────────────────────────────────────────────────────────────────────
def filter_product_single_generic(entities):
    result = []
    for e in entities:
        if e["label"] == "product":
            words = e["text"].strip().split()
            if len(words) == 1 and words[0].lower() in _PRODUCT_SINGLE_BLOCKLIST:
                continue
        result.append(e)
    return result


# ────────────────────────────────────────────────────────────────────────────
# Step 12 — Containment deduplication  ← BIGGEST single precision gain
#
# Problem: regex fires "June 12th" AND "Thursday, June 12th" for the same date.
# Both pass through as separate entities, counting as 2 dates for 1 event.
#
# Fix: within each label group, if entity A's text is a proper substring of
# entity B's text, drop A (keep the longer, more specific span).
# ────────────────────────────────────────────────────────────────────────────
def deduplicate_with_containment(entities):
    by_label = defaultdict(list)
    for e in entities:
        by_label[e["label"]].append(e)
    result = []
    for label, ents in by_label.items():
        ents_sorted = sorted(ents, key=lambda x: len(x["text"]), reverse=True)
        kept = []
        for e in ents_sorted:
            tl = e["text"].lower().strip()
            covered = any(
                tl in k["text"].lower().strip() and tl != k["text"].lower().strip()
                for k in kept
            )
            if not covered:
                kept.append(e)
        result.extend(kept)
    return result


print("✓ All 12 post-processing functions defined")

## 6 · Individual Entity Extractors

In [ ]:
def extract_emails(text):
    return [{"label": "email address", "text": m.group(), "score": 1.0}
            for m in _EMAIL_RE.finditer(text)]

def extract_phones(text):
    res, seen = [], set()
    for m in _PHONE_RE.finditer(text):
        t = m.group().strip()
        if len(re.sub(r'\D','',t)) >= 7 and t not in seen:
            res.append({"label": "phone number", "text": t, "score": 1.0}); seen.add(t)
    return res

def extract_money(text):
    res, seen = [], set()
    for pat in _MONEY_PATTERNS:
        for m in pat.finditer(text):
            t = m.group().strip()
            if t not in seen:
                res.append({"label": "money", "text": t, "score": 0.95}); seen.add(t)
    return res

def extract_dates(text):
    res, seen = [], set()
    for pat in _DATE_PATTERNS:
        for m in pat.finditer(text):
            t = m.group().strip()
            if t not in seen and len(t) > 2:
                res.append({"label": "date", "text": t, "score": 0.90}); seen.add(t)
    return res

def extract_competitors(text, gliner_ents):
    companies = {e["text"].lower().strip() for e in gliner_ents
                 if e["label"] in ("company", "competitor")}
    found, res = set(), []
    def add(name):
        name = name.strip().rstrip(".,;")
        if len(name) < 2 or name.lower() in found: return
        found.add(name.lower())
        res.append({"label": "competitor", "text": name, "score": 0.85})
    for pat in _COMP_PATTERNS:
        for m in pat.finditer(text):
            for gname in ["name", "name2"]:
                try:
                    c = m.group(gname)
                    if c:
                        c = c.strip().rstrip(" .,;")
                        if c.lower() in companies or any(k in c.lower() for k in _KNOWN_COMPETITORS):
                            add(c)
                except IndexError:
                    pass
    text_lower = text.lower()
    for kw in _KNOWN_COMPETITORS:
        if kw in text_lower:
            idx = text_lower.find(kw)
            add(text[idx:idx+len(kw)])
    return res

print("✓ Extractor functions defined")

## 7 · Full Extraction Pipeline

The `extract()` function runs a single document through the complete 12-step pipeline:

In [ ]:
def extract(text):
    """
    Run the full CRM entity extraction pipeline on a single text.
    Returns a list of entity dicts: {label, text, score}
    """
    # 1. Ensemble GLiNER — both models at low threshold (post-process will filter)
    ents_lg = model_lg.predict_entities(text, GLINER_LABELS, threshold=0.30)
    ents_md = model_md.predict_entities(text, GLINER_LABELS, threshold=0.30)
    gliner_raw = ents_lg + ents_md

    # 2. Per-label threshold
    gliner_filtered = apply_label_thresholds(gliner_raw)
    # Strip email/phone/competitor — handled by regex/rules
    gliner_core = [e for e in gliner_filtered
                   if e["label"] not in ("email address", "phone number", "competitor")]

    # 3. Regex layers
    emails    = extract_emails(text)
    phones    = extract_phones(text)
    money_r   = extract_money(text)
    dates_r   = extract_dates(text)
    comp      = extract_competitors(text, gliner_filtered)

    # 4. Combine everything
    combined = gliner_core + emails + phones + money_r + dates_r + comp

    # 5–12. Post-processing chain
    combined = deduplicate(combined)
    combined = merge_adjacent_spans(combined, text)
    combined = cross_entity_validate(combined)
    combined = filter_products_by_context(combined, text)
    combined = filter_pronoun_companies(combined)
    combined = filter_person_pronouns(combined)
    combined = filter_money_no_digits(combined)
    combined = filter_location_refined(combined)
    combined = filter_job_title_nontitles(combined)
    combined = filter_product_single_generic(combined)
    combined = deduplicate_with_containment(combined)
    extracted = deduplicate(combined)

    return extracted


print("✓ extract() pipeline defined")

### 7.1 — Live Demo

Let's run the full pipeline on our sample email and see the structured output:

In [ ]:
print("Input:")
print(DEMO_TEXT)
print()

t0 = time.time()
entities = extract(DEMO_TEXT)
elapsed = (time.time() - t0) * 1000

# Display as a nice table
rows = [{"Label": e["label"], "Extracted Text": e["text"],
         "Confidence": f"{e.get('score', 1.0):.2f}"}
        for e in sorted(entities, key=lambda x: x["label"])]

print(f"\nExtracted {len(entities)} entities in {elapsed:.0f}ms:\n")
df_demo = pd.DataFrame(rows)
display(df_demo.style
    .set_caption("Extracted Entities")
    .set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

## 8 · Test Suite

The benchmark uses **150 annotated test cases** covering:
- 31 email happy path  · 31 email edge case  · 15 email failure mode  
- 26 chat happy path  · 26 chat edge case  · 21 chat failure mode

Each case has a `text` (raw input) and `expected` (ground truth entity dict).

**Subcategories:**
| Subcategory | Description |
|---|---|
| `happy_path` | Clean, well-structured inputs — the easy cases |
| `edge_case` | Ambiguous, partial, or unusual language |
| `failure_mode` | Cases designed to stress-test the model (over/under extraction) |

In [ ]:
# ── Load test cases (embedded compressed data) ──────────────────────────────
import base64, zlib, json

_CASES_DATA = "..."

_raw = json.loads(zlib.decompress(base64.b64decode(_CASES_DATA)).decode("utf-8"))

@dataclass
class TestCase:
    id: str
    category: str
    subcategory: str
    description: str
    text: str
    expected: dict
    notes: str = ""

TEST_CASES = [TestCase(**c) for c in _raw]
print(f"✓ Loaded {len(TEST_CASES)} test cases")

# Summary
counts = Counter((c.category, c.subcategory) for c in TEST_CASES)
rows = [{"Category": k[0], "Subcategory": k[1], "Count": v}
        for k, v in sorted(counts.items())]
display(pd.DataFrame(rows))

## 9 · Scoring

For each test case we compute **recall** and **precision** against the ground truth.

**Recall** = how many expected entities did we find?  
**Precision** = of everything we extracted, how much was correct?

A match is counted when:  
`expected_value.lower() in extracted_text.lower()`  
— i.e. the expected value is a *substring* of what we extracted (tolerates extra context).

In [ ]:
def score(extracted, expected):
    if not expected:
        # No expected entities → any extraction is a hallucination
        return {
            "recall": None, "precision": None,
            "over_extraction": len(extracted) > 0,
            "hits": [], "misses": [],
            "hallucinations": [(e["label"], e["text"], round(e.get("score",1.0),2))
                               for e in extracted]
        }
    flat = [(l, v) for l, vs in expected.items() for v in vs]
    hits, misses, halls = [], [], []
    for l, v in flat:
        if any(e["label"]==l and v.lower() in e["text"].lower() for e in extracted):
            hits.append((l, v))
        else:
            misses.append((l, v))
    for e in extracted:
        if not any(e["label"]==l and v.lower() in e["text"].lower() for l,v in flat):
            halls.append((e["label"], e["text"], round(e.get("score",1.0), 2)))
    r = len(hits) / len(flat) if flat else 1.0
    p = len(hits) / len(extracted) if extracted else 1.0
    return {"recall": r, "precision": p, "over_extraction": False,
            "hits": hits, "misses": misses, "hallucinations": halls}

print("✓ Scoring function defined")

## 10 · Run the Benchmark

This cell runs the full pipeline on all 150 test cases.  
⏱️ *Expected time: ~10–15 minutes on Colab CPU (2 GLiNER models × 150 cases)*

In [ ]:
from IPython.display import clear_output

all_results = []
n = len(TEST_CASES)

for i, tc in enumerate(TEST_CASES):
    t0 = time.time()
    extracted = extract(tc.text)
    elapsed_ms = (time.time() - t0) * 1000
    sc = score(extracted, tc.expected)

    all_results.append({
        "id": tc.id,
        "category": tc.category,
        "subcategory": tc.subcategory,
        "description": tc.description,
        "elapsed_ms": round(elapsed_ms, 1),
        "extracted": extracted,
        "hits": sc["hits"],
        "misses": sc["misses"],
        "hallucinations": sc["hallucinations"],
        "recall": sc["recall"],
        "precision": sc["precision"],
    })

    # Progress update every 10 cases
    if (i+1) % 10 == 0 or (i+1) == n:
        scored = [r for r in all_results if r["recall"] is not None]
        avg_r = sum(r["recall"] for r in scored) / len(scored) if scored else 0
        avg_p = sum(r["precision"] for r in scored) / len(scored) if scored else 0
        clear_output(wait=True)
        print(f"Progress: {i+1}/{n} cases  "
              f"Avg Recall={avg_r:.1%}  Avg Precision={avg_p:.1%}")

print(f"\n✓ Benchmark complete — {n} cases processed")

## 11 · Results

In [ ]:
# ── Overall metrics ──────────────────────────────────────────────────────────
scored = [r for r in all_results if r["recall"] is not None]
avg_r = sum(r["recall"] for r in scored) / len(scored)
avg_p = sum(r["precision"] for r in scored) / len(scored)
f1 = 2 * avg_r * avg_p / (avg_r + avg_p)
avg_ms = sum(r["elapsed_ms"] for r in all_results) / len(all_results)
total_halls = sum(len(r["hallucinations"]) for r in all_results)

# Per-subcategory
sub_metrics = {}
for sub in ["happy_path", "edge_case", "failure_mode"]:
    s = [r for r in scored if r["subcategory"] == sub]
    sub_metrics[sub] = {
        "recall": sum(r["recall"] for r in s) / len(s) if s else 0,
        "precision": sum(r["precision"] for r in s) / len(s) if s else 0,
        "count": len(s)
    }

# Display summary table
summary_data = [
    {"Metric": "Overall Recall",    "Value": f"{avg_r:.1%}", "Target": "≥ 95%",
     "Status": "✅" if avg_r >= 0.95 else "⚠️"},
    {"Metric": "Overall Precision", "Value": f"{avg_p:.1%}", "Target": "≥ 85%",
     "Status": "✅" if avg_p >= 0.85 else "⚠️"},
    {"Metric": "F1 Score",          "Value": f"{f1:.3f}",   "Target": "≥ 0.90",
     "Status": "✅" if f1 >= 0.90 else "⚠️"},
    {"Metric": "Happy Path Recall", "Value": f"{sub_metrics['happy_path']['recall']:.1%}", "Target": "≥ 95%",
     "Status": "✅" if sub_metrics['happy_path']['recall'] >= 0.95 else "⚠️"},
    {"Metric": "Edge Case Recall",  "Value": f"{sub_metrics['edge_case']['recall']:.1%}",  "Target": "≥ 90%",
     "Status": "✅" if sub_metrics['edge_case']['recall'] >= 0.90 else "⚠️"},
    {"Metric": "Failure Mode Recall","Value": f"{sub_metrics['failure_mode']['recall']:.1%}","Target": "≥ 85%",
     "Status": "✅" if sub_metrics['failure_mode']['recall'] >= 0.85 else "⚠️"},
    {"Metric": "Avg Latency",       "Value": f"{avg_ms:.0f}ms", "Target": "< 1500ms",
     "Status": "✅" if avg_ms < 1500 else "⚠️"},
    {"Metric": "Total Hallucinations","Value": str(total_halls), "Target": "< 100",
     "Status": "✅" if total_halls < 100 else "⚠️"},
]

df_summary = pd.DataFrame(summary_data)
display(HTML("<h3>Overall Benchmark Results</h3>"))
display(df_summary.style.set_properties(**{"text-align":"left"})
        .hide(axis="index"))

### 11.1 — Per-Label Recall

In [ ]:
# Compute per-label recall
label_hits, label_total = {}, {}
for r in all_results:
    for l, _ in r.get("hits", []):
        label_hits[l] = label_hits.get(l, 0) + 1
        label_total[l] = label_total.get(l, 0) + 1
    for l, _ in r.get("misses", []):
        label_total[l] = label_total.get(l, 0) + 1

labels_sorted = sorted(label_total, key=lambda l: -(label_hits.get(l,0)/label_total[l]))
recalls = [label_hits.get(l,0)/label_total[l] for l in labels_sorted]
totals  = [label_total[l] for l in labels_sorted]
hits_   = [label_hits.get(l,0) for l in labels_sorted]

colors = ["#2ecc71" if r >= 0.90 else "#f39c12" if r >= 0.70 else "#e74c3c"
          for r in recalls]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(labels_sorted, recalls, color=colors, edgecolor="white", height=0.65)

# Annotate bars
for bar, r, h, t in zip(bars, recalls, hits_, totals):
    ax.text(min(r + 0.01, 0.99), bar.get_y() + bar.get_height()/2,
            f"{r:.0%}  ({h}/{t})", va="center", fontsize=10, fontweight="bold")

ax.set_xlim(0, 1.15)
ax.set_xlabel("Recall", fontsize=12)
ax.set_title("Per-Label Recall — Hybrid NER v3", fontsize=14, fontweight="bold")
ax.axvline(0.95, color="#e74c3c", linestyle="--", alpha=0.6, label="95% target")
ax.axvline(0.90, color="#f39c12", linestyle="--", alpha=0.4, label="90% line")

legend_patches = [
    mpatches.Patch(color="#2ecc71", label="≥ 90% (green)"),
    mpatches.Patch(color="#f39c12", label="70–90% (amber)"),
    mpatches.Patch(color="#e74c3c", label="< 70% (red)"),
]
ax.legend(handles=legend_patches, loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

# Print as table
df_label = pd.DataFrame([
    {"Label": l, "Recall": f"{label_hits.get(l,0)/label_total[l]:.0%}",
     "Hits": label_hits.get(l,0), "Total": label_total[l]}
    for l in labels_sorted
])
display(df_label.style.hide(axis="index"))

### 11.2 — Subcategory Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

subs = ["happy_path", "edge_case", "failure_mode"]
labels_x = ["Happy Path", "Edge Case", "Failure Mode"]
rec_vals  = [sub_metrics[s]["recall"]    for s in subs]
prec_vals = [sub_metrics[s]["precision"] for s in subs]

x = np.arange(len(subs))
w = 0.35

# Recall by subcategory
ax = axes[0]
b1 = ax.bar(x - w/2, rec_vals,  w, label="Recall",    color="#3498db", alpha=0.85)
b2 = ax.bar(x + w/2, prec_vals, w, label="Precision", color="#e67e22", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels_x, fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(0.95, color="red",    linestyle="--", alpha=0.5, label="95% recall target")
ax.axhline(0.85, color="orange", linestyle="--", alpha=0.5, label="85% precision target")
ax.set_title("Recall & Precision by Subcategory", fontweight="bold")
ax.legend(fontsize=9)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f"{bar.get_height():.1%}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Hallucination breakdown
hall_by_label = Counter()
for r in all_results:
    for label, _, _ in r.get("hallucinations", []):
        hall_by_label[label] += 1

ax2 = axes[1]
if hall_by_label:
    hall_labels = [k for k, v in hall_by_label.most_common()]
    hall_counts = [v for k, v in hall_by_label.most_common()]
    bar_colors  = plt.cm.Reds(np.linspace(0.4, 0.8, len(hall_labels)))
    ax2.barh(hall_labels[::-1], hall_counts[::-1], color=bar_colors[::-1])
    ax2.set_xlabel("Count", fontsize=11)
    ax2.set_title(f"Remaining Hallucinations by Label\n(total = {sum(hall_counts)})",
                  fontweight="bold")
    for i, (cnt, lbl) in enumerate(zip(hall_counts[::-1], hall_labels[::-1])):
        ax2.text(cnt + 0.3, i, str(cnt), va="center", fontsize=10)

plt.tight_layout()
plt.show()

### 11.3 — Version Comparison (Baseline → v1 → v2 → v3)

In [ ]:
comparison_data = {
    "Metric":       ["Avg Recall", "Avg Precision", "F1 Score",
                     "Happy Path", "Edge Case", "Failure Mode",
                     "Hallucinations", "Latency"],
    "Baseline":     ["84%", "80%", "0.820", "88%", "84%", "74%", "—",    "340ms"],
    "Hybrid v1":    ["91%", "82%", "0.864", "94%", "91%", "87%", "—",    "680ms"],
    "Hybrid v2":    ["96.7%","77%","0.859", "98.9%","97.4%","88.9%","209","884ms"],
    "Hybrid v3 ✅": [f"{avg_r:.1%}", f"{avg_p:.1%}", f"{f1:.3f}",
                     f"{sub_metrics['happy_path']['recall']:.1%}",
                     f"{sub_metrics['edge_case']['recall']:.1%}",
                     f"{sub_metrics['failure_mode']['recall']:.1%}",
                     str(total_halls), f"{avg_ms:.0f}ms"],
}
df_cmp = pd.DataFrame(comparison_data)

def highlight_v3(s):
    return ["" if col != "Hybrid v3 ✅" else "background-color: #d4edda; font-weight: bold"
            for col in s.index]

display(HTML("<h4>Version Comparison Table</h4>"))
display(df_cmp.style.apply(highlight_v3, axis=1).hide(axis="index"))

### 11.4 — Sample Results (First 10 Cases)

In [ ]:
sample_rows = []
for r in all_results[:10]:
    if r["recall"] is None:
        continue
    entities_str = "; ".join(
        f"{e['label']}: {e['text'][:30]}"
        for e in r["extracted"][:4]
    ) + ("..." if len(r["extracted"]) > 4 else "")
    sample_rows.append({
        "ID": r["id"],
        "Type": r["subcategory"].replace("_"," "),
        "Recall": f"{r['recall']:.0%}",
        "Precision": f"{r['precision']:.0%}",
        "Hallucinations": len(r["hallucinations"]),
        "Sample extractions": entities_str
    })

df_sample = pd.DataFrame(sample_rows)
display(df_sample.style
    .set_properties(subset=["Sample extractions"], **{"text-align":"left", "font-size":"11px"})
    .hide(axis="index"))

## 12 · Try It Yourself

Enter any sales email or CRM chat message below and run the cell to see what the pipeline extracts:

In [ ]:
YOUR_TEXT = """
Hi,

I'm Layla Hassan, procurement lead at Gulf Dynamics. We have around 800 employees
spread across UAE, Saudi Arabia, and Egypt. Our current CRM is Salesforce but we're
evaluating alternatives. Budget is around AED 600,000 annually.

Can we schedule a demo this Thursday or next Monday?

Best,
Layla
layla.hassan@gulfdynamics.ae | +971 55 900 1234
"""

# Run extraction
t0 = time.time()
result = extract(YOUR_TEXT)
elapsed = (time.time() - t0) * 1000

print(f"Extracted {len(result)} entities in {elapsed:.0f}ms:\n")
df_your = pd.DataFrame([
    {"Label": e["label"], "Text": e["text"], "Score": f"{e.get('score',1.0):.2f}"}
    for e in sorted(result, key=lambda x: x["label"])
])
display(df_your.style.set_properties(**{"text-align":"left"}).hide(axis="index"))